Melanoma Classification — Model Training & Comparison
**Models compared:** Simple CNN (from scratch) vs ResNet18 (transfer learning)

**Research question:** Does transfer learning improve melanoma classification over a custom CNN trained from scratch?

Notebook flow:
1. Setup — imports, device, load data from notebook 01
2. Model definitions — SimpleCNN and ResNet18
3. Training loop — shared, identical for both models
4. Train SimpleCNN → save best checkpoint
5. Train ResNet18 → save best checkpoint
6. Evaluate both on the held-out test set
7. Confusion matrices, ROC curves, metrics table


## 1 · Imports & device

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import random
import copy
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from torchvision.transforms import InterpolationMode
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
)
import seaborn as sns

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device — supports CUDA (NVIDIA), MPS (Apple Silicon), or CPU ──────────────
device = (
    torch.device("mps")  if torch.backends.mps.is_available()  else
    torch.device("cuda") if torch.cuda.is_available()          else
    torch.device("cpu")
)
print(f"Using device: {device}")
print(device)
print(torch.backends.mps.is_available())


Using device: mps
mps
True


2 · Load data
All settings are identical — same seed, same split, same transforms.


In [2]:
# ── Config (must match notebook 01) ──────────────────────────────────────────
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_WORKERS  = 2
PIN_MEMORY   = torch.cuda.is_available()
VAL_FRACTION = 0.5
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

BASE_DIR  = Path().resolve()
DATA_DIR  = BASE_DIR          # notebook lives inside melanoma_cancer_dataset/
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR  = DATA_DIR / "test"

assert TRAIN_DIR.exists(), f"train/ not found at {TRAIN_DIR}"
assert TEST_DIR.exists(),  f"test/  not found at {TEST_DIR}"

# ── Transforms ────────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20),
                       interpolation=InterpolationMode.BILINEAR),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20),
                       interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Datasets ──────────────────────────────────────────────────────────────────
def stratified_split(dataset, val_fraction, seed):
    rng = random.Random(seed)
    class_indices = {}
    for idx, (_, label) in enumerate(dataset.samples):
        class_indices.setdefault(label, []).append(idx)
    val_idx, test_idx = [], []
    for label, indices in class_indices.items():
        indices = indices.copy()
        rng.shuffle(indices)
        split = int(len(indices) * val_fraction)
        val_idx.extend(indices[:split])
        test_idx.extend(indices[split:])
    return val_idx, test_idx

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
test_pool     = datasets.ImageFolder(TEST_DIR,  transform=eval_transform)

class_names  = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

val_indices, test_indices = stratified_split(test_pool, VAL_FRACTION, SEED)
val_dataset  = Subset(test_pool, val_indices)
test_dataset = Subset(test_pool, test_indices)

# ── Class weights ─────────────────────────────────────────────────────────────
train_labels = [label for _, label in train_dataset.samples]
label_counts = Counter(train_labels)
n_total, n_classes = len(train_labels), len(label_counts)
class_weights = torch.tensor(
    [n_total / (n_classes * label_counts[i]) for i in range(n_classes)],
    dtype=torch.float32,
)

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f"Classes      : {class_to_idx}")
print(f"Train        : {len(train_dataset)} images")
print(f"Val          : {len(val_dataset)} images")
print(f"Test         : {len(test_dataset)} images  ← never touched during training")
print(f"Class weights: {class_weights.tolist()}")


Classes      : {'benign': 0, 'malignant': 1}
Train        : 9605 images
Val          : 500 images
Test         : 500 images  ← never touched during training
Class weights: [0.9605000019073486, 1.0428881645202637]


## 3 · Model definitions

### Why these two models?

**Simple CNN (from scratch)**
A 3-block convolutional network with no prior knowledge. It learns everything
from the melanoma images alone. This is our *lower bound* — if transfer learning
doesn't beat this, something is wrong.

**ResNet18 (transfer learning)**
Pretrained on ImageNet (1.4 M images, 1 000 classes). The backbone already
knows low-level features (edges, textures, colour gradients) that are also
present in dermoscopy images. We freeze the backbone first, train only the
new classifier head, then unfreeze the last residual block for fine-tuning.
This is the standard two-stage transfer learning strategy.

The key difference being tested: **does prior knowledge from natural images
transfer usefully to medical skin images?**


In [3]:
# ── Simple CNN ────────────────────────────────────────────────────────────────
class SimpleCNN(nn.Module):
    """
    3-block CNN trained from random weights.
    Architecture: Conv→ReLU→Pool ×3, then AdaptiveAvgPool→Dropout→Linear.
    AdaptiveAvgPool makes it input-size agnostic (works with any IMG_SIZE).
    """
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1 — learn basic edges and colour gradients
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 2 — combine edges into simple shapes
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 3 — combine shapes into higher-level lesion features
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),   # collapse spatial dims → (batch, 64, 1, 1)
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ── ResNet18 ──────────────────────────────────────────────────────────────────
def build_resnet18(num_classes=2, freeze_backbone=True):
    """
    ResNet18 pretrained on ImageNet.
    - freeze_backbone=True  → only the new fc layer trains (Stage 1)
    - freeze_backbone=False → full network trains (Stage 2 fine-tune)
    """
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Replace the final layer — original outputs 1000 ImageNet classes
    in_features = model.fc.in_features        # 512 for ResNet18
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, num_classes),
    )
    return model


def unfreeze_resnet_layer4(model):
    """Unfreeze the last residual block and classifier for fine-tuning."""
    for param in model.layer4.parameters():
        param.requires_grad = True
    for param in model.fc.parameters():
        param.requires_grad = True
    return model


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Quick param count check
cnn     = SimpleCNN(num_classes=2)
resnet  = build_resnet18(freeze_backbone=True)

print(f"SimpleCNN    trainable params : {count_trainable_params(cnn):>10,}")
print(f"ResNet18     trainable params : {count_trainable_params(resnet):>10,}  (head only, backbone frozen)")
resnet_full = build_resnet18(freeze_backbone=False)
print(f"ResNet18     total params     : {count_trainable_params(resnet_full):>10,}  (all unfrozen)")


SimpleCNN    trainable params :     25,954
ResNet18     trainable params :      1,026  (head only, backbone frozen)
ResNet18     total params     : 11,177,538  (all unfrozen)


## 4 · Training configuration

In [4]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
CNN_EPOCHS          = 15     # SimpleCNN trains for 15 epochs
RESNET_FROZEN_EPOCHS    = 5  # Stage 1: backbone frozen, train head only
RESNET_FINETUNE_EPOCHS  = 10 # Stage 2: unfreeze layer4, lower learning rate
CNN_LR              = 1e-3
RESNET_FROZEN_LR    = 1e-3
RESNET_FINETUNE_LR  = 1e-4   # 10× smaller for fine-tuning

Path("models").mkdir(exist_ok=True)
Path("reports").mkdir(exist_ok=True)

# Criterion — class-weighted cross-entropy
# With our balanced dataset weights ≈ 1.0, but good practice to always use it
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

print("Training config:")
print(f"  SimpleCNN   : {CNN_EPOCHS} epochs  lr={CNN_LR}")
print(f"  ResNet18    : {RESNET_FROZEN_EPOCHS} frozen epochs (lr={RESNET_FROZEN_LR})")
print(f"              + {RESNET_FINETUNE_EPOCHS} fine-tune epochs (lr={RESNET_FINETUNE_LR})")
print(f"  Loss        : CrossEntropyLoss(weight={class_weights.tolist()})")
print(f"  Device      : {device}")


Training config:
  SimpleCNN   : 15 epochs  lr=0.001
  ResNet18    : 5 frozen epochs (lr=0.001)
              + 10 fine-tune epochs (lr=0.0001)
  Loss        : CrossEntropyLoss(weight=[0.9605000019073486, 1.0428881645202637])
  Device      : mps


## 5 · Shared training & evaluation functions
Both models use the **exact same** training loop so the comparison is fair.
The only thing that differs is the model architecture and learning rate.

**Best-model saving strategy:** we save the checkpoint whenever **val recall
(sensitivity) improves**, not val loss. For melanoma classification, missing a
malignant lesion (false negative) is clinically far more dangerous than a false
alarm (false positive). Optimising recall directly ensures the saved model
minimises missed malignancies.


In [5]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """
    Returns loss, accuracy, recall, AUC, all_labels, all_probs.
    Recall = sensitivity = TP / (TP + FN) for the malignant class.
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += images.size(0)
            probs = torch.softmax(outputs, dim=1)[:, 1]   # malignant probability
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    loss     = total_loss / total
    accuracy = correct / total
    preds    = [1 if p >= 0.5 else 0 for p in all_probs]
    recall   = recall_score(all_labels, preds, pos_label=1, zero_division=0)
    auc      = roc_auc_score(all_labels, all_probs)
    return loss, accuracy, recall, auc, all_labels, all_probs


def train_model(model, train_loader, val_loader, optimizer, criterion,
                scheduler, device, epochs, save_path, model_name,
                start_epoch=0, history=None):
    """
    Train for `epochs` epochs.
    Saves best checkpoint by val RECALL (most critical metric for cancer detection).
    Returns history dict and best val recall achieved.
    """
    if history is None:
        history = {"train_loss": [], "train_acc": [],
                   "val_loss": [],   "val_acc": [],
                   "val_recall": [], "val_auc": []}

    best_val_recall = 0.0
    best_weights    = None

    for epoch in range(1, epochs + 1):
        global_epoch = start_epoch + epoch

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_recall, val_auc, _, _ = evaluate(
            model, val_loader, criterion, device)

        if scheduler is not None:
            scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_recall"].append(val_recall)
        history["val_auc"].append(val_auc)

        # Save best model by val recall
        marker = ""
        if val_recall > best_val_recall:
            best_val_recall = val_recall
            best_weights    = copy.deepcopy(model.state_dict())
            torch.save({
                "epoch":          global_epoch,
                "model_state":    best_weights,
                "val_recall":     val_recall,
                "val_auc":        val_auc,
                "class_to_idx":   class_to_idx,
            }, save_path)
            marker = "  ← best recall saved"

        print(f"[{model_name}] Epoch {global_epoch:02d} | "
              f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
              f"val loss {val_loss:.4f} acc {val_acc:.3f} "
              f"recall {val_recall:.3f} AUC {val_auc:.3f}{marker}")

    # Restore best weights for returning
    if best_weights is not None:
        model.load_state_dict(best_weights)

    return history, best_val_recall


print("Training functions defined.")


Training functions defined.


## 6 · Train Simple CNN
Trained end-to-end from random initialisation. No prior knowledge — every
feature must be learned from the 9 605 training images alone.


In [6]:
cnn_model = SimpleCNN(num_classes=2).to(device)

cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=CNN_LR)
cnn_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    cnn_optimizer, mode="min", patience=3, factor=0.5, verbose=True
)

print(f"SimpleCNN trainable params: {count_trainable_params(cnn_model):,}")
print(f"Training for {CNN_EPOCHS} epochs...\n")

cnn_history, cnn_best_recall = train_model(
    model        = cnn_model,
    train_loader = train_loader,
    val_loader   = val_loader,
    optimizer    = cnn_optimizer,
    criterion    = criterion,
    scheduler    = cnn_scheduler,
    device       = device,
    epochs       = CNN_EPOCHS,
    save_path    = "models/simple_cnn_best.pth",
    model_name   = "SimpleCNN",
)

print(f"\nSimpleCNN training done. Best val recall: {cnn_best_recall:.4f}")


SimpleCNN trainable params: 25,954
Training for 15 epochs...



/opt/anaconda3/envs/engg2112/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


[SimpleCNN] Epoch 01 | train loss 0.5246 acc 0.743 | val loss 0.3754 acc 0.822 recall 0.724 AUC 0.925  ← best recall saved
[SimpleCNN] Epoch 02 | train loss 0.4774 acc 0.776 | val loss 0.3791 acc 0.822 recall 0.680 AUC 0.935
[SimpleCNN] Epoch 03 | train loss 0.4543 acc 0.792 | val loss 0.3198 acc 0.858 recall 0.800 AUC 0.945  ← best recall saved
[SimpleCNN] Epoch 04 | train loss 0.4256 acc 0.810 | val loss 0.3115 acc 0.848 recall 0.760 AUC 0.948
[SimpleCNN] Epoch 05 | train loss 0.4162 acc 0.813 | val loss 0.3350 acc 0.848 recall 0.860 AUC 0.935  ← best recall saved
[SimpleCNN] Epoch 06 | train loss 0.3904 acc 0.828 | val loss 0.2884 acc 0.874 recall 0.872 AUC 0.952  ← best recall saved
[SimpleCNN] Epoch 07 | train loss 0.3930 acc 0.828 | val loss 0.2868 acc 0.866 recall 0.824 AUC 0.951
[SimpleCNN] Epoch 08 | train loss 0.3815 acc 0.832 | val loss 0.3077 acc 0.852 recall 0.828 AUC 0.940
[SimpleCNN] Epoch 09 | train loss 0.3749 acc 0.838 | val loss 0.2832 acc 0.860 recall 0.816 AUC 0.95

## 7 · Train ResNet18 (two-stage transfer learning)

**Stage 1 — frozen backbone (5 epochs)**
Only the new classifier head trains. The pretrained ImageNet weights are
preserved. This is fast and avoids corrupting good features with a high
learning rate before the head is stable.

**Stage 2 — fine-tune layer4 (10 epochs)**
The last residual block (`layer4`) is unfrozen alongside the head, using a
10× smaller learning rate. This lets the deeper features adapt to dermoscopy
images without catastrophic forgetting of the ImageNet knowledge.


In [7]:
# ── Stage 1: train head only ─────────────────────────────────────────────────
resnet_model = build_resnet18(num_classes=2, freeze_backbone=True).to(device)

resnet_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet_model.parameters()),
    lr=RESNET_FROZEN_LR,
)
resnet_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    resnet_optimizer, mode="min", patience=3, factor=0.5, verbose=True
)

print(f"Stage 1 — backbone frozen")
print(f"Trainable params: {count_trainable_params(resnet_model):,}  (head only)")
print(f"Training for {RESNET_FROZEN_EPOCHS} epochs...\n")

resnet_history, _ = train_model(
    model        = resnet_model,
    train_loader = train_loader,
    val_loader   = val_loader,
    optimizer    = resnet_optimizer,
    criterion    = criterion,
    scheduler    = resnet_scheduler,
    device       = device,
    epochs       = RESNET_FROZEN_EPOCHS,
    save_path    = "models/resnet18_best.pth",
    model_name   = "ResNet18-S1",
)

# ── Stage 2: unfreeze layer4, lower learning rate ────────────────────────────
resnet_model = unfreeze_resnet_layer4(resnet_model)

resnet_ft_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet_model.parameters()),
    lr=RESNET_FINETUNE_LR,
)
resnet_ft_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    resnet_ft_optimizer, T_max=RESNET_FINETUNE_EPOCHS
)

print(f"\nStage 2 — layer4 + head unfrozen")
print(f"Trainable params: {count_trainable_params(resnet_model):,}")
print(f"Training for {RESNET_FINETUNE_EPOCHS} epochs...\n")

resnet_history, resnet_best_recall = train_model(
    model        = resnet_model,
    train_loader = train_loader,
    val_loader   = val_loader,
    optimizer    = resnet_ft_optimizer,
    criterion    = criterion,
    scheduler    = resnet_ft_scheduler,
    device       = device,
    epochs       = RESNET_FINETUNE_EPOCHS,
    save_path    = "models/resnet18_best.pth",
    model_name   = "ResNet18-S2",
    start_epoch  = RESNET_FROZEN_EPOCHS,
    history      = resnet_history,
)

print(f"\nResNet18 training done. Best val recall: {resnet_best_recall:.4f}")


Stage 1 — backbone frozen
Trainable params: 1,026  (head only)
Training for 5 epochs...



/opt/anaconda3/envs/engg2112/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


[ResNet18-S1] Epoch 01 | train loss 0.4217 acc 0.803 | val loss 0.2872 acc 0.888 recall 0.896 AUC 0.953  ← best recall saved
[ResNet18-S1] Epoch 02 | train loss 0.3818 acc 0.835 | val loss 0.2962 acc 0.880 recall 0.916 AUC 0.951  ← best recall saved
[ResNet18-S1] Epoch 03 | train loss 0.3772 acc 0.837 | val loss 0.2800 acc 0.882 recall 0.868 AUC 0.949
[ResNet18-S1] Epoch 04 | train loss 0.3728 acc 0.835 | val loss 0.3077 acc 0.860 recall 0.932 AUC 0.953  ← best recall saved
[ResNet18-S1] Epoch 05 | train loss 0.3720 acc 0.840 | val loss 0.3404 acc 0.850 recall 0.944 AUC 0.954  ← best recall saved

Stage 2 — layer4 + head unfrozen
Trainable params: 8,394,754
Training for 10 epochs...

[ResNet18-S2] Epoch 06 | train loss 0.2972 acc 0.881 | val loss 0.2276 acc 0.898 recall 0.936 AUC 0.976  ← best recall saved


/opt/anaconda3/envs/engg2112/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


[ResNet18-S2] Epoch 07 | train loss 0.2365 acc 0.907 | val loss 0.1977 acc 0.914 recall 0.888 AUC 0.975


...

/opt/anaconda3/envs/engg2112/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


[ResNet18-S2] Epoch 14 | train loss 0.1535 acc 0.940 | val loss 0.2140 acc 0.908 recall 0.888 AUC 0.978


/opt/anaconda3/envs/engg2112/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


[ResNet18-S2] Epoch 15 | train loss 0.1484 acc 0.943 | val loss 0.1943 acc 0.920 recall 0.892 AUC 0.980

ResNet18 training done. Best val recall: 0.9360


/opt/anaconda3/envs/engg2112/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


## 8 · Training curves

In [8]:
def plot_training_curves(history, title, save_path):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"{title} — Training Curves", fontsize=13)

    axes[0].plot(epochs, history["train_loss"], label="Train")
    axes[0].plot(epochs, history["val_loss"],   label="Val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], label="Train")
    axes[1].plot(epochs, history["val_acc"],   label="Val")
    axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()

    axes[2].plot(epochs, history["val_recall"], label="Val Recall", color="tomato")
    axes[2].plot(epochs, history["val_auc"],    label="Val AUC",    color="purple")
    axes[2].set_title("Recall & AUC (val)"); axes[2].set_xlabel("Epoch"); axes[2].legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Saved → {save_path}")

plot_training_curves(cnn_history,    "Simple CNN", "reports/cnn_training_curves.png")
plot_training_curves(resnet_history, "ResNet18",   "reports/resnet_training_curves.png")


Saved → reports/cnn_training_curves.png
Saved → reports/resnet_training_curves.png


/var/folders/x2/j0llhrdj7jdcb8yckrnm24vm0000gn/T/ipykernel_78280/858647220.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9 · Load best checkpoints & evaluate on test set
We now load the **best checkpoint** (saved by val recall) for each model
and run it once on the held-out test set.
The test set was never used during training or validation — these numbers
are an unbiased estimate of real-world performance.


In [9]:
def load_best_model(model, path, device):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    saved_epoch  = checkpoint.get("epoch", "?")
    saved_recall = checkpoint.get("val_recall", "?")
    print(f"  Loaded {path}  (epoch {saved_epoch}, val recall {saved_recall:.4f})")
    model.eval()
    return model

print("Loading best checkpoints...")
cnn_model    = load_best_model(SimpleCNN(num_classes=2).to(device),
                                "models/simple_cnn_best.pth", device)
resnet_model = load_best_model(build_resnet18(num_classes=2, freeze_backbone=False).to(device),
                                "models/resnet18_best.pth", device)

# Evaluate on test set
print("\nEvaluating on test set...")
_, cnn_acc,    cnn_recall,    cnn_auc,    cnn_labels,    cnn_probs    = evaluate(cnn_model,    test_loader, criterion, device)
_, resnet_acc, resnet_recall, resnet_auc, resnet_labels, resnet_probs = evaluate(resnet_model, test_loader, criterion, device)

print(f"\nSimpleCNN  — acc {cnn_acc:.4f}  recall {cnn_recall:.4f}  AUC {cnn_auc:.4f}")
print(f"ResNet18   — acc {resnet_acc:.4f}  recall {resnet_recall:.4f}  AUC {resnet_auc:.4f}")


/var/folders/x2/j0llhrdj7jdcb8yckrnm24vm0000gn/T/ipykernel_78280/1384287206.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=de

Loading best checkpoints...
  Loaded models/simple_cnn_best.pth  (epoch 13, val recall 0.9240)
  Loaded models/resnet18_best.pth  (epoch 6, val recall 0.9360)

Evaluating on test set...

SimpleCNN  — acc 0.8000  recall 0.9320  AUC 0.9324
ResNet18   — acc 0.9040  recall 0.9160  AUC 0.9713


## 10 · Confusion matrices
Rows = true labels, Columns = predicted labels.

**Key cells for melanoma:**
- **FN (bottom-left)** = malignant predicted as benign → most dangerous error
- **TP (bottom-right)** = malignant correctly detected → what we maximise


In [10]:
def get_predictions(probs, threshold=0.5):
    return [1 if p >= threshold else 0 for p in probs]

THRESHOLD = 0.5

cnn_preds    = get_predictions(cnn_probs,    THRESHOLD)
resnet_preds = get_predictions(resnet_probs, THRESHOLD)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"Confusion Matrices — Test Set (threshold={THRESHOLD})", fontsize=13)

for ax, labels, preds, title in [
    (axes[0], cnn_labels,    cnn_preds,    "Simple CNN"),
    (axes[1], resnet_labels, resnet_preds, "ResNet18"),
]:
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax,
                annot_kws={"size": 14})
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual",    fontsize=11)
    ax.set_title(title,        fontsize=12)

    # Annotate FN cell
    tn, fp, fn, tp = cm.ravel()
    ax.text(0.5, 1.5, f"FN={fn}\n(missed malignant)",
            ha="center", va="center", color="red", fontsize=9, style="italic")

plt.tight_layout()
plt.savefig("reports/confusion_matrices.png", dpi=150)
plt.show()
print("Saved → reports/confusion_matrices.png")


Saved → reports/confusion_matrices.png


/var/folders/x2/j0llhrdj7jdcb8yckrnm24vm0000gn/T/ipykernel_78280/3497989932.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11 · ROC curves

In [11]:
fig, ax = plt.subplots(figsize=(7, 6))

for labels, probs, name, color in [
    (cnn_labels,    cnn_probs,    "Simple CNN", "steelblue"),
    (resnet_labels, resnet_probs, "ResNet18",   "tomato"),
]:
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{name}  (AUC = {auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random classifier")
ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
ax.set_ylabel("True Positive Rate (Sensitivity / Recall)", fontsize=11)
ax.set_title("ROC Curve Comparison — Test Set", fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig("reports/roc_comparison.png", dpi=150)
plt.show()
print("Saved → reports/roc_comparison.png")


Saved → reports/roc_comparison.png


/var/folders/x2/j0llhrdj7jdcb8yckrnm24vm0000gn/T/ipykernel_78280/2339155478.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12 · Threshold sweep
The default threshold of 0.5 is arbitrary. For melanoma we may prefer a **lower
threshold** (e.g. 0.3) to catch more malignant cases at the cost of more false alarms.
This table shows the tradeoff at common thresholds.


In [12]:
def threshold_table(labels, probs, model_name):
    print(f"\n{model_name}")
    print(f"{'Threshold':>10} {'Accuracy':>10} {'Precision':>10} "
          f"{'Recall':>8} {'F1':>8} {'FN':>5} {'FP':>5}")
    print("-" * 62)
    for t in [0.3, 0.35, 0.4, 0.45, 0.5]:
        preds = [1 if p >= t else 0 for p in probs]
        cm    = confusion_matrix(labels, preds, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        print(f"{t:>10.2f} "
              f"{accuracy_score(labels, preds):>10.4f} "
              f"{precision_score(labels, preds, pos_label=1, zero_division=0):>10.4f} "
              f"{recall_score(labels, preds, pos_label=1, zero_division=0):>8.4f} "
              f"{f1_score(labels, preds, pos_label=1, zero_division=0):>8.4f} "
              f"{fn:>5} {fp:>5}")

threshold_table(cnn_labels,    cnn_probs,    "Simple CNN")
threshold_table(resnet_labels, resnet_probs, "ResNet18")



Simple CNN
 Threshold   Accuracy  Precision   Recall       F1    FN    FP
--------------------------------------------------------------
      0.30     0.7420     0.6685   0.9600   0.7882    10   119
      0.35     0.7700     0.6957   0.9600   0.8067    10   105
      0.40     0.7800     0.7096   0.9480   0.8116    13    97
      0.45     0.7860     0.7200   0.9360   0.8139    16    91
      0.50     0.8000     0.7373   0.9320   0.8233    17    83

ResNet18
 Threshold   Accuracy  Precision   Recall       F1    FN    FP
--------------------------------------------------------------
      0.30     0.8740     0.8281   0.9440   0.8822    14    49
      0.35     0.8820     0.8473   0.9320   0.8876    17    42
      0.40     0.8880     0.8619   0.9240   0.8919    19    37
      0.45     0.8940     0.8774   0.9160   0.8963    21    32
      0.50     0.9040     0.8945   0.9160   0.9051    21    27


## 13 · Final metrics comparison

In [13]:
def full_metrics(labels, probs, threshold=0.5):
    preds = [1 if p >= threshold else 0 for p in probs]
    cm    = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "Accuracy":  accuracy_score(labels, preds),
        "Precision": precision_score(labels, preds, pos_label=1, zero_division=0),
        "Recall":    recall_score(labels, preds, pos_label=1, zero_division=0),
        "F1":        f1_score(labels, preds, pos_label=1, zero_division=0),
        "AUC":       roc_auc_score(labels, probs),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
    }

cnn_metrics    = full_metrics(cnn_labels,    cnn_probs)
resnet_metrics = full_metrics(resnet_labels, resnet_probs)

print("=" * 62)
print(f"FINAL TEST SET RESULTS  (threshold = 0.5)")
print("=" * 62)
print(f"{'Metric':<12} {'Simple CNN':>14} {'ResNet18':>14} {'Winner':>10}")
print("-" * 62)
for metric in ["Accuracy", "Precision", "Recall", "F1", "AUC"]:
    cv = cnn_metrics[metric]
    rv = resnet_metrics[metric]
    winner = "ResNet18" if rv > cv else "SimpleCNN" if cv > rv else "tie"
    print(f"{metric:<12} {cv:>14.4f} {rv:>14.4f} {winner:>10}")

print("-" * 62)
for metric in ["TP", "TN", "FP", "FN"]:
    cv = cnn_metrics[metric]
    rv = resnet_metrics[metric]
    print(f"{metric:<12} {cv:>14} {rv:>14}")

print("\nFN = malignant images incorrectly classified as benign (missed cancer)")
print("Lower FN = safer model for clinical use")

# Classification reports
print("\n── Simple CNN ───────────────────────────────────────────────")
print(classification_report(cnn_labels, get_predictions(cnn_probs),
                             target_names=class_names, zero_division=0))

print("── ResNet18 ─────────────────────────────────────────────────")
print(classification_report(resnet_labels, get_predictions(resnet_probs),
                             target_names=class_names, zero_division=0))


FINAL TEST SET RESULTS  (threshold = 0.5)
Metric           Simple CNN       ResNet18     Winner
--------------------------------------------------------------
Accuracy             0.8000         0.9040   ResNet18
Precision            0.7373         0.8945   ResNet18
Recall               0.9320         0.9160  SimpleCNN
F1                   0.8233         0.9051   ResNet18
AUC                  0.9324         0.9713   ResNet18
--------------------------------------------------------------
TP                      233            229
TN                      167            223
FP                       83             27
FN                       17             21

FN = malignant images incorrectly classified as benign (missed cancer)
Lower FN = safer model for clinical use

── Simple CNN ───────────────────────────────────────────────
              precision    recall  f1-score   support

      benign       0.91      0.67      0.77       250
   malignant       0.74      0.93      0.82       25

## 14 · Visual metrics comparison

In [14]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
cnn_vals    = [cnn_metrics[m]    for m in metrics_to_plot]
resnet_vals = [resnet_metrics[m] for m in metrics_to_plot]

x = np.arange(len(metrics_to_plot))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, cnn_vals,    w, label="Simple CNN", color="steelblue")
b2 = ax.bar(x + w/2, resnet_vals, w, label="ResNet18",   color="tomato")

ax.set_xticks(x); ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylim(0, 1.1); ax.set_ylabel("Score"); 
ax.set_title("Model Comparison — Test Set Metrics", fontsize=13)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.5)
ax.legend(fontsize=11)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("reports/metrics_comparison.png", dpi=150)
plt.show()
print("Saved → reports/metrics_comparison.png")


Saved → reports/metrics_comparison.png


/var/folders/x2/j0llhrdj7jdcb8yckrnm24vm0000gn/T/ipykernel_78280/1928316061.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
